In [ ]:
from aeon.transformations.collection.interval_based import SupervisedIntervals, RandomIntervals
from tscglue.interval_models import UnsupervisedIntervals

t = UnsupervisedIntervals(n_jobs=-1)
s = SupervisedIntervals(n_jobs=-1)
r = RandomIntervals(n_intervals=600, n_jobs=-1)

In [3]:
from tscglue import data_loader


dataset = "LargeKitchenAppliances"
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)

In [4]:
#print("UnsupervisedIntervals:", t.fit_transform(X_train).shape)

In [5]:
from sklearn.random_projection import GaussianRandomProjection
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

X_unsup_train = t.fit_transform(X_train)
X_unsup_test = t.transform(X_test)
rp = GaussianRandomProjection(random_state=42)
rp.fit(X_unsup_train)
X_unsup_rp_train = rp.transform(X_unsup_train)
X_unsup_rp_test = rp.transform(X_unsup_test)

X_sup_train = s.fit_transform(X_train, y_train)
X_sup_test = s.transform(X_test)

X_rand_train = r.fit_transform(X_train)
X_rand_test = r.transform(X_test)

# Concatenate UnsupIntervals + GaussianRP with RandomIntervals
X_combined_train = np.hstack((X_unsup_rp_train, X_rand_train))
X_combined_test = np.hstack((X_unsup_rp_test, X_rand_test))

print(f"UnsupIntervals + RP: {X_unsup_rp_train.shape[1]} features")
print(f"SupervisedIntervals: {X_sup_train.shape[1]} features")
print(f"RandomIntervals:     {X_rand_train.shape[1]} features")
print(f"UnsupRP + Random:    {X_combined_train.shape[1]} features")

datasets = [
    ("UnsupIntervals + GaussianRP", X_unsup_rp_train, X_unsup_rp_test),
    ("SupervisedIntervals",         X_sup_train,      X_sup_test),
    ("RandomIntervals",             X_rand_train,     X_rand_test),
    ("UnsupRP + RandomIntervals",   X_combined_train, X_combined_test),
]

classifiers = [
    ("ExtraTrees",         lambda: ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=42)),
    ("ExtraTrees+PCA(95%)", lambda: make_pipeline(PCA(n_components=0.95), ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=42))),
    ("RidgeCV",            lambda: RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
]

for clf_name, clf_fn in classifiers:
    print(f"\n--- {clf_name} ---")
    for name, Xtr, Xte in datasets:
        clf = clf_fn()
        clf.fit(Xtr, y_train)
        acc = accuracy_score(y_test, clf.predict(Xte))
        print(f"{name:30s} -> accuracy: {acc:.4f}")

UnsupIntervals + RP: 5080 features
SupervisedIntervals: 6478 features
RandomIntervals:     4200 features
UnsupRP + Random:    9280 features

--- ExtraTrees ---
UnsupIntervals + GaussianRP    -> accuracy: 0.4507
SupervisedIntervals            -> accuracy: 0.6133
RandomIntervals                -> accuracy: 0.6293
UnsupRP + RandomIntervals      -> accuracy: 0.6133

--- ExtraTrees+PCA(95%) ---
UnsupIntervals + GaussianRP    -> accuracy: 0.4053
SupervisedIntervals            -> accuracy: 0.4240
RandomIntervals                -> accuracy: 0.5707
UnsupRP + RandomIntervals      -> accuracy: 0.4800

--- RidgeCV ---
UnsupIntervals + GaussianRP    -> accuracy: 0.4373
SupervisedIntervals            -> accuracy: 0.4507
RandomIntervals                -> accuracy: 0.5813
UnsupRP + RandomIntervals      -> accuracy: 0.4480


In [6]:
#print("RandomIntervals:", r.fit_transform(X_train).shape)